# Safe Evidence — treino do YOLOv9 (violência x não violência) no Colab

Antes de rodar: **Ambiente de execução → Alterar tipo → GPU (T4)**.

Etapas: clonar o repo → baixar XD-Violence → extrair frames → treinar → avaliar → testar num vídeo.
Os pesos finais vão para o Google Drive para não se perderem quando o Colab desligar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/safe_evidence_ia'
!mkdir -p $DRIVE

## 1. Código
Se o repositório for privado, crie um token (Contents: read) e use `https://<TOKEN>@github.com/...`.

In [ ]:
REPO = 'https://github.com/DanielleSilvaPereira/Projeto_YOLO_violencia.git'
!git clone $REPO /content/projeto
%cd /content/projeto
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Dataset XD-Violence
O conjunto de teste (800 vídeos) tem anotação **por frame** — é a fonte principal dos frames violentos.
Dos vídeos de treino, baixamos só uma amostra de vídeos **normais** (`label_A`) para ter mais exemplos negativos.

O download completo tem dezenas de GB; comece com o subconjunto abaixo.

In [ ]:
!python scripts/baixar_xd_violence.py --destino data/xd_violence --partes test_videos --normais-treino 300

## 3. Extrair frames rotulados
- Categorias padrão: luta (B1), tiro (B2), tumulto (B4), abuso (B5). Acidente de carro (B6) e explosão (G) ficam de fora.
- Divisão por vídeo: 70% treino, 15% validação, 15% teste.
- Adicione `--usar-fracos` se tiver baixado vídeos violentos de treino (rótulo só por vídeo).

In [ ]:
!python scripts/preparar_dataset.py --formato xd \
    --videos data/xd_violence --anotacoes data/xd_violence/test_annotations.txt \
    --saida data/frames --fps 1 --max-frames-por-video 64 --balancear --workers 4
!cat data/frames/resumo.json

## 4. Treinar YOLOv9-C (classificação)
O backbone vem pré-treinado no COCO (`yolov9c.pt`, baixado automaticamente).
Na T4, `--lote 64 --imgsz 224` cabe na memória.

In [ ]:
!python scripts/treinar.py --dados data/frames --epocas 30 --lote 64 --imgsz 224
!cp pesos/melhor.pt pesos/melhor.json $DRIVE/
!cp -r runs/violencia $DRIVE/runs

### (Opcional) baseline para comparar no TCC

In [ ]:
# !python scripts/treinar.py --dados data/frames --modelo yolo11n-cls.pt --nome yolo11n-cls --epocas 30

## 5. Avaliar (métricas por frame e por vídeo + RN16)

In [ ]:
!python scripts/avaliar.py --pesos pesos/melhor.pt --dados data/frames --por-video --saida resultados
!cp -r resultados $DRIVE/
from IPython.display import Image, display
display(Image('resultados/matriz_frames.png')); display(Image('resultados/matriz_videos.png'))

## 6. Testar num vídeo qualquer

In [ ]:
from google.colab import files
up = files.upload()
video = list(up)[0]
!python scripts/analisar_video.py "$video" --video-saida anotado.mp4 --grafico linha.png
display(Image('linha.png'))